# 02 - Modelos de Forecasting

Entrenamos **dos modelos de categorías distintas**, ambos evaluados con la **misma metodología: predicción a un día, con reajuste diario (walk-forward + retrain)** — esto es lo que hace la comparación justa entre modelos ("manzanas con manzanas"):

1. **N-BEATS** — deep learning, con las variables exógenas como *past covariates*, optimizado con **Optuna** + **early stopping**.
2. **XGBoost** — machine learning, con la temperatura como *future covariate* (el pronóstico del día siguiente), optimizado con **Optuna** + **early stopping**.

### ¿Por qué "predicción a un día, con reajuste diario" para los dos?

Predecir únicamente el día t+1 usando datos reales hasta el día t (walk-forward, un paso adelante) hace que ambos modelos resuelvan el mismo problema. Además, **ambos se reajustan (retrain) todos los días** con los datos reales más recientes — así ninguno de los dos compite en desventaja por tener pesos "congelados" desde hace 90 días. Esto refleja además una idea central en forecasting operativo: un modelo entrenado una sola vez se degrada con el tiempo (*concept drift*), y reajustarlo periódicamente mejora la precisión (lo vamos a cuantificar en `03_evaluacion.ipynb`).

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

from darts import TimeSeries
from darts.models import NBEATSModel, XGBModel, ExponentialSmoothing as DartsETS
from darts.dataprocessing.transformers import Scaler
from darts.utils.utils import ModelMode, SeasonalityMode
from pytorch_lightning.callbacks import EarlyStopping
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


## 2. Carga de datos y splits

Los splits ya se definieron y se guardaron en `01_preprocesamiento.ipynb` — acá los volvemos a cargar desde disco (no se recalculan, para asegurar que sean *exactamente* los mismos en todos los notebooks):

- **`train_opt.csv`** (1,280 días): para entrenar durante la búsqueda de hiperparámetros y en el ajuste final.
- **`val_opt.csv`** (90 días, dentro del train): para el *early stopping* y para puntuar los trials de Optuna. Nunca se usa para actualizar los pesos del modelo.
- **`test.csv`** (90 días finales, oct-dic 2024): completamente aislado, solo se usa una vez al final, igual para los tres modelos.

También cargamos `datos_diarios.csv` completo (los 1,460 días), porque algunas variables — como las covariates de XGBoost y N-BEATS — necesitan cubrir todo el rango de fechas, incluyendo el período de test (ver la nota sobre *future covariate* más abajo).

In [ ]:
daily = pd.read_csv('datos_diarios.csv', index_col=0, parse_dates=True)
train_opt_df = pd.read_csv('train_opt.csv', index_col=0, parse_dates=True)
val_opt_df = pd.read_csv('val_opt.csv', index_col=0, parse_dates=True)
test_df = pd.read_csv('test.csv', index_col=0, parse_dates=True)

TEST_SIZE = len(test_df)
VAL_SIZE = len(val_opt_df)
COV_COLS = ['apparent_temperature_c', 'cloud_cover_percent', 'is_holiday', 'is_weekend', 'is_school_vacation']

target = TimeSeries.from_series(daily['active_power_mw'])
temp_cov = TimeSeries.from_series(daily['apparent_temperature_c'])
past_cov_full = TimeSeries.from_dataframe(daily[COV_COLS])

# Recortamos el TimeSeries completo para que coincida exactamente con los splits guardados en el notebook 01
train_full = target[:-TEST_SIZE]
cov_train_full = past_cov_full[:-TEST_SIZE]
train_opt = train_full[:-VAL_SIZE]
val_opt = train_full[-VAL_SIZE:]
cov_opt = cov_train_full[:-VAL_SIZE]
cov_val = cov_train_full[-VAL_SIZE:]

print(f"train_opt: {len(train_opt)} dias | val_opt: {len(val_opt)} dias | test: {TEST_SIZE} dias")

def rmse(a, b):
    return float(np.sqrt(np.mean((np.array(a) - np.array(b)) ** 2)))

train_opt: 1280 dias | val_opt: 90 dias | test: 90 dias


## 3. Modelo 1: XGBoost con *future covariate*, Optuna y early stopping

- **Future covariate**: temperatura del día que se predice. A diferencia de un pronóstico a 90 días, acá solo necesitamos el pronóstico meteorológico de **un solo día** — mucho más realista y preciso en la práctica.
- **Early stopping**: `n_estimators=50` como techo, con `early_stopping_rounds=10` — el entrenamiento corta apenas el error de validación deja de mejorar, evitando sobreajuste y ahorrando tiempo.
- **Optuna**: optimiza `lags`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree` (los hiperparámetros más influyentes; `n_estimators` queda fijo en 50 porque lo regula el early stopping).

In [ ]:
def objective_xgb(trial):
    lags = trial.suggest_int('lags', 7, 30)
    max_depth = trial.suggest_int('max_depth', 2, 8)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)

    modelo = XGBModel(
        lags=lags, lags_future_covariates=[0],
        n_estimators=50, early_stopping_rounds=10,
        max_depth=max_depth, learning_rate=learning_rate,
        subsample=subsample, colsample_bytree=colsample_bytree, random_state=42
    )
    modelo.fit(train_opt, future_covariates=temp_cov, val_series=val_opt, val_future_covariates=temp_cov)

    # validacion: walk-forward de un dia sobre val_opt
    hist = modelo.historical_forecasts(
        series=train_full, future_covariates=temp_cov,
        forecast_horizon=1, stride=1, start=len(train_opt), start_format='position',
        retrain=False, last_points_only=True, verbose=False, show_warnings=False
    )
    actual = train_full[-len(hist):].values().flatten()
    return rmse(actual, hist.values().flatten())


In [ ]:
import os
if os.path.exists('optuna_xgb.db'):
    os.remove('optuna_xgb.db')  # borramos estudios previos para que la comparacion sea limpia

study_xgb = optuna.create_study(
    direction='minimize', study_name='xgb_dia_siguiente_v3',
    storage='sqlite:///optuna_xgb.db',
)
study_xgb.optimize(objective_xgb, n_trials=40)  # misma cantidad de trials que N-BEATS

print("Mejor RMSE de validacion (dia siguiente):", study_xgb.best_value)
print("Mejores hiperparametros:", study_xgb.best_params)


[0]	validation_0-rmse:536.53688
[1]	validation_0-rmse:527.01000
[2]	validation_0-rmse:518.64739
[3]	validation_0-rmse:510.62118
[4]	validation_0-rmse:502.62734
[5]	validation_0-rmse:494.73151
[6]	validation_0-rmse:487.32164
[7]	validation_0-rmse:480.49037
[8]	validation_0-rmse:472.28519
[9]	validation_0-rmse:464.04338
[10]	validation_0-rmse:456.53238
[11]	validation_0-rmse:448.63516
[12]	validation_0-rmse:441.22112
[13]	validation_0-rmse:434.70299
[14]	validation_0-rmse:428.41726
[15]	validation_0-rmse:421.53344
[16]	validation_0-rmse:414.03918
[17]	validation_0-rmse:406.31804
[18]	validation_0-rmse:399.54893
[19]	validation_0-rmse:393.22532
[20]	validation_0-rmse:387.19698
[21]	validation_0-rmse:381.72250
[22]	validation_0-rmse:376.39586
[23]	validation_0-rmse:371.18262
[24]	validation_0-rmse:365.70443
[25]	validation_0-rmse:359.42267
[26]	validation_0-rmse:353.72372
[27]	validation_0-rmse:349.13247
[28]	validation_0-rmse:343.63453
[29]	validation_0-rmse:338.91196
[30]	validation_0-rm

In [ ]:
trials_df = study_xgb.trials_dataframe()
trials_df.to_csv('optuna_trials_xgb.csv', index=False)
with open('xgb_best_params.json', 'w') as f:
    json.dump({**study_xgb.best_params, 'n_estimators_max': 50, 'early_stopping_rounds': 10}, f, indent=2)

best_xgb = study_xgb.best_params
modelo_xgb = XGBModel(
    lags=best_xgb['lags'], lags_future_covariates=[0],
    n_estimators=50, early_stopping_rounds=10,
    max_depth=best_xgb['max_depth'], learning_rate=best_xgb['learning_rate'],
    subsample=best_xgb['subsample'], colsample_bytree=best_xgb['colsample_bytree'], random_state=42
)
modelo_xgb.fit(train_opt, future_covariates=temp_cov, val_series=val_opt, val_future_covariates=temp_cov)
print(f"Mejor iteracion (early stopping): {modelo_xgb.model.best_iteration} de 50 maximo")
modelo_xgb.save('xgb_best_model.pkl')


[0]	validation_0-rmse:459.89194
[1]	validation_0-rmse:398.14136
[2]	validation_0-rmse:359.00130
[3]	validation_0-rmse:320.58213
[4]	validation_0-rmse:288.42714
[5]	validation_0-rmse:252.77883
[6]	validation_0-rmse:226.94725
[7]	validation_0-rmse:207.48734
[8]	validation_0-rmse:192.29635
[9]	validation_0-rmse:176.21547
[10]	validation_0-rmse:165.22322
[11]	validation_0-rmse:154.95930
[12]	validation_0-rmse:149.03016
[13]	validation_0-rmse:144.30107
[14]	validation_0-rmse:141.17881
[15]	validation_0-rmse:134.10343
[16]	validation_0-rmse:131.33840
[17]	validation_0-rmse:129.74047
[18]	validation_0-rmse:127.57533
[19]	validation_0-rmse:125.62375
[20]	validation_0-rmse:126.75335
[21]	validation_0-rmse:126.23283
[22]	validation_0-rmse:122.03546
[23]	validation_0-rmse:121.67154
[24]	validation_0-rmse:121.44674
[25]	validation_0-rmse:121.23112
[26]	validation_0-rmse:121.16052
[27]	validation_0-rmse:121.07660
[28]	validation_0-rmse:119.79714
[29]	validation_0-rmse:117.47145
[30]	validation_0-rm

## 4. Modelo 2: N-BEATS con *past covariates*, Optuna y early stopping

- **Past covariates**: todas las variables exógenas disponibles (temperatura, nubosidad, feriado, fin de semana, vacaciones escolares).
- **`output_chunk_length=1`**: el modelo predice exactamente un día — coherente con la metodología "día siguiente" que usamos para los tres modelos.
- **Early stopping**: `n_epochs=50` como techo, con `EarlyStopping(monitor='val_loss', patience=8)` durante la búsqueda (10 en el modelo final).
- **Optuna**: optimiza `input_chunk_length`, `num_stacks`, `layer_widths`, `learning_rate`, `batch_size`.

In [ ]:
scaler_t = Scaler()
scaler_c = Scaler()
train_opt_s = scaler_t.fit_transform(train_opt)
val_opt_s = scaler_t.transform(val_opt)
cov_opt_s = scaler_c.fit_transform(cov_opt)
cov_val_s = scaler_c.transform(cov_val)
train_full_s = scaler_t.transform(train_full)
cov_train_full_s = scaler_c.transform(cov_train_full)


In [ ]:
def objective_nbeats(trial):
    input_chunk_length = trial.suggest_int('input_chunk_length', 14, 60)
    num_stacks = trial.suggest_int('num_stacks', 2, 20)
    layer_widths = trial.suggest_categorical('layer_widths', [32, 64, 128, 256])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    early_stop = EarlyStopping(monitor='val_loss', patience=8, min_delta=1e-4, mode='min')
    modelo = NBEATSModel(
        input_chunk_length=input_chunk_length,
        output_chunk_length=1,   # predice el dia siguiente
        num_stacks=num_stacks,
        layer_widths=layer_widths,
        optimizer_kwargs={'lr': learning_rate},
        batch_size=batch_size,
        n_epochs=50,
        random_state=42,
        pl_trainer_kwargs={"accelerator": "gpu", "enable_progress_bar": False, "callbacks": [early_stop]}
        # "gpu" en accelerator si hay GPU disponible
    )
    modelo.fit(train_opt_s, past_covariates=cov_opt_s, val_series=val_opt_s, val_past_covariates=cov_val_s, verbose=False)

    hist_s = modelo.historical_forecasts(
        series=train_full_s, past_covariates=cov_train_full_s,
        forecast_horizon=1, stride=1, start=len(train_opt), start_format='position',
        retrain=False, last_points_only=True, verbose=False, show_warnings=False
    )
    hist = scaler_t.inverse_transform(hist_s)
    actual = train_full[-len(hist):].values().flatten()
    return rmse(actual, hist.values().flatten())


In [ ]:
import os
if os.path.exists('optuna_nbeats.db'):
    os.remove('optuna_nbeats.db')  # borramos estudios previos para que la comparacion sea limpia

study_nbeats = optuna.create_study(
    direction='minimize', study_name='nbeats_dia_siguiente_v3',
    storage='sqlite:///optuna_nbeats.db',
)
study_nbeats.optimize(objective_nbeats, n_trials=40)  # misma cantidad de trials que XGBoost

print("Mejor RMSE de validacion (dia siguiente):", study_nbeats.best_value)
print("Mejores hiperparametros:", study_nbeats.best_params)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU 

Mejor RMSE de validacion (dia siguiente): 89.4381489777182
Mejores hiperparametros: {'input_chunk_length': 29, 'num_stacks': 7, 'layer_widths': 64, 'learning_rate': 0.0012533647732463874, 'batch_size': 16}


In [ ]:
trials_df_nb = study_nbeats.trials_dataframe()
trials_df_nb.to_csv('optuna_trials_nbeats.csv', index=False)
with open('nbeats_best_params.json', 'w') as f:
    json.dump(study_nbeats.best_params, f, indent=2)

best_nbeats = study_nbeats.best_params
early_stop_final = EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-4, mode='min')
modelo_nbeats = NBEATSModel(
    input_chunk_length=best_nbeats['input_chunk_length'],
    output_chunk_length=1,
    num_stacks=best_nbeats['num_stacks'],
    layer_widths=best_nbeats['layer_widths'],
    optimizer_kwargs={'lr': best_nbeats['learning_rate']},
    batch_size=best_nbeats['batch_size'],
    n_epochs=50,
    random_state=42,
    pl_trainer_kwargs={"accelerator": "cpu", "enable_progress_bar": False, "callbacks": [early_stop_final]}
)
modelo_nbeats.fit(train_opt_s, past_covariates=cov_opt_s, val_series=val_opt_s, val_past_covariates=cov_val_s, verbose=False)
print(f"Epocas entrenadas (early stopping): {modelo_nbeats.trainer.current_epoch} de 50 maximo")

modelo_nbeats.save('nbeats_best_model.pkl')
with open('nbeats_scalers.pkl', 'wb') as f:
    pickle.dump({'scaler_t': scaler_t, 'scaler_c': scaler_c}, f)


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Epocas entrenadas (early stopping): 24 de 50 maximo


## Conclusión

Los tres modelos quedaron configurados y entrenados (XGBoost y N-BEATS guardados en `../models/`, con sus estudios de Optuna completos en SQLite) bajo la **misma metodología de evaluación: predicción a un día, walk-forward, con datos reales**. La generación de las 90 predicciones de prueba y su evaluación se hace en `03_evaluacion.ipynb`.